In [ ]:
import yaml
import json
from IPython.display import display, HTML

# def read_all_yaml_files(path: str) -> List[Dict[str, Any]]:
#   result = []
#   for file in glob.iglob(path):
#     with open(file, 'r') as fh:
#       result.append(yaml.safe_load(fh))
#   return result

# def merge_yaml_schema(schema: List[Dict[str, Any]]) -> Dict[str, Any]:
#   result = {}
#   for s in schema:
#     print("parsing...")
#     for k, v in s.items():
#       if k not in result:
#         result[k] = v
#       elif isinstance(v, dict):
#         result[k] = merge_yaml_schema([result[k], v])
#       else:
#         # result[k] = type(v)([result[k], v])
#         result[k] = [result[k], v]
#   return result

def merge_values(data1, data2):
  """
  Recursively merge values from two YAML data structures.
  Lists are concatenated, and dictionaries are merged.
  """
  if isinstance(data1, dict) and isinstance(data2, dict):
    for key2, value2 in data2.items():
      if key2 in data1:
        value1 = data1[key2]
        if isinstance(value1, dict) != isinstance(value2, dict):
          if isinstance(value1, dict):
            value2 = {"##no-key": value2}
          else:
            value1 = {"##no-key": value1}
        data1[key2] = merge_values(value1, value2)
      else:
        data1[key2] = value2
    return data1
  elif isinstance(data1, list) and isinstance(data2, list):
    return data1 + data2
  else:
    if data1 is None:
      res = data2
    elif isinstance(data1, list):
      uniq_vals = set(data1 + [data2])
      res = list(sorted(uniq_vals, key=lambda x: (x is None, x)))
    else:
      res = sorted([data1, data2], key=lambda x: (x is None, x))
    return res

# barracks_yaml_files = read_all_yaml_files(r'S:\src\unknown-horizons\content\objects\**\*.yaml')
from pathlib import Path

files = list(Path(r's:\src\unknown-horizons\content\objects').glob(r'**\*.yaml'))
res = {}
for file in files:
  print(f"Merging {file}")
  with open(file, 'r') as fh:
    obj = yaml.safe_load(fh)
    res = merge_values(res, obj)
display(HTML("<pre>"+json.dumps(res, indent=2)+"</pre>"))
print("Done")

In [3]:
from pathlib import Path, PurePosixPath
from jinja2 import Environment, FileSystemLoader
from PIL import Image, ImageOps
import yaml

def rt(template_str: str, args: dict):
  env = Environment(loader=FileSystemLoader("."), trim_blocks=True) # trim blocks remove extra newline around for loops and other blocks
  template = env.from_string(template_str.strip())
  rendered_content = template.render(args)
  return rendered_content
def write_template(output_path: Path, template_str: str, args: dict):
  print(f"Writing template: {output_path}")
  rendered_content = rt(template_str, args)
  output_path.write_text(rendered_content)
def generate_uid(building_name: str, file_id: str) -> str: # generates valid Godot UID for the given building name and file id
  valid_uid: str = "c"
  # add the building name to the uid
  stripped_building_name: str = building_name.lower().replace("_", "").replace("z", "").replace("9", "") # strip the building name of unallowed characters
  building_name_cutted: str = stripped_building_name[:min(len(stripped_building_name), 10)] # cut the building name to leave room for the first letter of id
  valid_uid += building_name_cutted
  # add the file id to the uid
  stripped_file_id = file_id.replace("_", "").lower().replace("z", "").replace("9", "") # strip the file id of unallowed characters
  file_id_cutted = stripped_file_id[:min(len(stripped_file_id), 13 - len(valid_uid))] # cut the file id to fit the remaining space in the uid to 13 characters
  valid_uid += file_id_cutted
  valid_uid = valid_uid.ljust(13, "0") # add a spacer to make the uid 13 characters
  print(f"Generated uid: {valid_uid}")
  return valid_uid

In [ ]:
bakery_tscn_template = """
[gd_scene load_steps=8 format=3 uid="uid://{{ tscn_uid }}"]

{% if baseclass.startswith("collectors.") %}
[ext_resource type="Script" uid="uid://dwc6g0dxao8e0" path="res://Assets/World/Components/Collectors/BuildingCollector/BuildingCollectorComponent.gd" id="1_script_gd"]
{% else %}
[ext_resource type="Script" uid="uid://4liotsbcpcls" path="res://Assets/World/Buildings/Building2D.gd" id="1_script_gd"]
{% endif %}
[ext_resource type="PackedScene" uid="uid://x1upwhg1f71a" path="res://Assets/World/Components/Selectable/Selectable.tscn" id="2_selectable_component"]
[ext_resource type="PackedScene" uid="uid://c7w3xnajww1kq" path="res://Assets/World/Components/BuildingActionSet/BuildingActionSet.tscn" id="3_building_action_set_component"]
[ext_resource type="SpriteFrames" uid="uid://{{ tres_uid }}" path="res://{{ sprite_frames_tres_path }}" id="4_sprite_frames"]
[ext_resource type="PackedScene" uid="uid://kv52rh351xud" path="res://Assets/World/Components/AmbientSoundComponent/AmbientSoundComponent.tscn" id="3_ambient_sound_component"]
[ext_resource type="PackedScene" uid="uid://b3hix4pdlnumi" path="res://Assets/World/Components/Storages/SizedStorageComponent/SizedStorageComponent.tscn" id="5_sized_storage_component"]
[ext_resource type="PackedScene" uid="uid://ck36mqeae5ana" path="res://Assets/World/Components/Storages/SlotStorageComponent/SlotStorageComponent.tscn" id="5_slot_storage_component"]
[ext_resource type="PackedScene" uid="uid://bo27kwd5m1jmc" path="res://Assets/World/Components/ProductionLine/ProductionLineComponent.tscn" id="6_production_line_component"]
[ext_resource type="PackedScene" uid="uid://dyqwsks6v6puj" path="res://Assets/World/Components/Collectors/BuildingCollector/BuildingCollectorComponent.tscn" id="7_building_collector_component"]

[node name="{{ name }}" type="Node2D"]
script = ExtResource("1_script_gd")
baseclass = "{{ baseclass | replace("collectors.", "") }}"
radius = {{ radius }}
{% if velocity is defined %}
velocity= {{ velocity }}
{% endif %}

{% if components.SelectableComponent is defined %}
[node name="Selectable" parent="." instance=ExtResource("2_selectable_component")]
type = "{{ components.SelectableComponent.type }}"
tabs = {{ components.SelectableComponent.tabs | tojson }}
enemy_tabs = {{ components.SelectableComponent.enemy_tabs | tojson  }}

{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.HealthComponent is defined %}
; [node name="HealthComponent" parent="." instance=ExtResource("2_health_component")]
; max_health = {{ components.HealthComponent.maxhealth }}

{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.ProducerComponent is defined %}
{% for line_name, line in components.ProducerComponent.productionlines.items() %}
[node name="ProductionLineComponent_{{ line_name }}" parent="." instance=ExtResource("6_production_line_component")]
line_name = "{{line_name}}"
produces = Dictionary[StringName, int]({{ line.produces | tojson | replace('"RES.', '&"') }})
consumes = Dictionary[StringName, int]({{ line.consumes | tojson | replace('"RES.', '&"') | replace(': -', ': ') }})

{% endfor %}
{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.StorageComponent is defined %}
{% if components.StorageComponent.SlotsStorage is defined %}
[node name="SlotStorageComponent" parent="." instance=ExtResource("5_slot_storage_component")]
max_capacity = Dictionary[StringName, int]({{ components.StorageComponent.SlotsStorage.slot_sizes | tojson | replace('"RES.', '&"') }})

{% endif %}
{% if components.StorageComponent.PositiveSizedSlotStorage is defined %}
[node name="SizedStorageComponent" parent="." instance=ExtResource("5_sized_storage_component")]
limit = {{ components.StorageComponent.PositiveSizedSlotStorage.limit }}

{% endif %}
{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.CollectingComponent is defined %}
[node name="BuildingCollectorComponent" parent="." instance=ExtResource("7_building_collector_component")]

{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.AmbientSoundComponent is defined and components.AmbientSoundComponent.soundfiles is defined %}
[node name="AmbientSoundComponent" parent="." instance=ExtResource("3_ambient_sound_component")]
sound_files = {{ components.AmbientSoundComponent.soundfiles | tojson }}
{% endif %}

[node name="BuildingActionSet" parent="." instance=ExtResource("3_building_action_set_component")]
sprite_frames = ExtResource("4_sprite_frames")

"""

animation_frames_tscn_template = """
[gd_resource type="SpriteFrames" load_steps=39 format=3 uid="uid://{{ uid }}"]

{% set unique_source_atlas_paths = sprite_frames_animations.values() | map(attribute='source_atlas_path') | list | unique %}
{% for fa in unique_source_atlas_paths %}
[ext_resource type="Texture2D" path="res://{{ fa }}" id="{{ fa.stem }}"]
{% endfor %}

{% for animation_name, sfa in sprite_frames_animations.items() %}
{% for frame_pos in sfa["animation_frames_positions"] %}
[sub_resource type="AtlasTexture" id="{{ animation_name }}_{{  frame_pos[0] }}_{{ frame_pos[1] }}"]
atlas = ExtResource("{{ sfa["source_atlas_path"].stem }}")
region = Rect2({{ frame_pos[0] }}, {{ frame_pos[1] }}, {{ frame_pos[2] }}, {{ frame_pos[3] }})
{% endfor %}
{% endfor %}

[resource]
animations = [
{% for animation_name, sfa in sprite_frames_animations.items() %}
  {
    "frames": [
    {% for frame_pos in sfa["animation_frames_positions"] %}
      {
        "duration": 1.0,
        "texture": SubResource("{{ animation_name }}_{{  frame_pos[0] }}_{{ frame_pos[1] }}")
      },
    {% endfor %}
    ],
    "loop": true,
    "name": &"{{ animation_name }}",
    "speed": 5.0
  },
{% endfor %}
]

"""

project_path = Path("s:/src/unknown-horizons-godot-port")
buildings_path = project_path / "Assets/World/Units2"

uh_original_buildings_path = Path(r"S:\src\unknown-horizons\content\objects\units")
# yaml_files = Path(uh_original_buildings_path.glob("warehouse.yaml")
# yaml_files = list(Path(uh_original_buildings_path.glob("pastryshop.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("tent.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("warehouse.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("lumberjackcamp.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("barracks.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("*.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("**/settlercollector.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("**/lumberjackcollector.yaml"))
yaml_files = list(uh_original_buildings_path.glob("**/*.yaml"))
# yaml_files = list(uh_original_buildings_path.glob("bakery.yaml"))
# gfx_buildings_path = (yaml_files[0].parent / "../../gfx/buildings").resolve()
gfx_path = (uh_original_buildings_path.parent.parent / "gfx").resolve()
# yaml_files = Path(r"S:\src\unknown-horizons\content\objects\buildings").glob("bakery.yaml")

def process_action_sets(action_sets: dict[str, dict[str, None]], building_name: str, save_path: Path):
  animations = {}
  for tier, as_names in action_sets.items():
    for as_name, _ in as_names.items():
      # print(tier, as_name)
      # action set folder is impossible to figure out based on the yaml file. Search for the corresponding folder:
      # as_folder = gfx_buildings_path / tier.removeprefix("TIER.").lower() / building_name / as_name
      as_folder_candidates = list(gfx_path.parent.glob(f"**/{as_name}"))
      assert len(as_folder_candidates) == 1 and as_folder_candidates[0].exists()
      as_folder = as_folder_candidates[0]
      # print(as_folder, as_folder.exists())
      # as_folder = Path(r"S:\src\unknown-horizons\content\gfx\buildings\pioneers\brewery\as_brewery0") # !!!
      diff = list(set(as_folder.glob("**/*.png")) - set(as_folder.glob("**/*.*")))
      if diff:
        raise Exception(f"UNEXPECTED diff: {diff}")
      for work_idle_path in as_folder.iterdir():
        if work_idle_path.stem == "deleteme": continue # some extra file
        sprite_data = {} # one sprite per type (WORK or IDLE or IDLE_FULL). Each sprite contains 4 rows of animation frames
        for angle_path in work_idle_path.iterdir():
          if angle_path.stem.startswith("tm_") and angle_path.is_file(): continue # some extra file
          angle = angle_path.stem.rjust(3, "0") # '45' => '045'
          sprite_data[angle] = sprite_datum = {
            "row_width": 0,
            "row_height": 0,
            "animation_frames": []
          }
          # sprite_datum["animation_frames"] = []
          for img_path in angle_path.iterdir():
            img = Image.open(img_path)
            sprite_datum["row_width"] += img.width
            sprite_datum["row_height"] = max(sprite_datum["row_height"], img.height)
            sprite_datum["animation_frames"].append({"path": img_path, "img": img})
            print(work_idle_path.stem, angle, img_path)
        total_width = max(sd["row_width"] for sd in sprite_data.values())
        total_height = sum(sd["row_height"] for sd in sprite_data.values())
        sprite_img = Image.new("RGBA", (total_width, total_height)) # each animation frame set on a separate row
        y = 0
        for angle, sprite_datum in sorted(sprite_data.items()):
          x = 0
          for frame in sprite_datum["animation_frames"]:
            sprite_img.paste(frame["img"], (x, y))
            frame["xywh"] = (x, y, *frame["img"].size)
            x += frame["img"].width
          y += sprite_datum["row_height"]
        # str(work_idle_path.relative_to(work_idle_path.parent.parent.parent)).replace("\\","_")
        file_name = (building_name + "_" + # eg. 'bakery_'
                    tier.removeprefix("TIER.").lower() + "_" +            # eg. 'sailors_'
                    as_name + "_" +                                       # eg. 'as_brewery0_'
                    work_idle_path.stem)                                  # eg. 'idle'
        file_path = save_path / f"{file_name}.png"
        print("Saving to ", file_path)
        sprite_img.save(file_path)
        # display(sprite_img)
        
        # save data for a single tres for all sprite frames for a particular building
        for angle, sprite_datum in sprite_data.items():
          animation_name = as_name.removeprefix("as_") + "." + tier.removeprefix("TIER.").lower() + "." + work_idle_path.stem + "." + angle
          if animation_name in animations:
            raise Exception(f"DUPLICATE ANIMATION NAME: {animation_name}")
          animations[animation_name] = {
            "source_atlas_path": PurePosixPath(file_path.relative_to(project_path)),
            "animation_frames_positions": [sd["xywh"] for sd in sprite_datum["animation_frames"]],
          }
        print(as_folder / f"{work_idle_path.stem}.png")

  return animations

buildings = [] # list of building infos
unique_uids = set()
for yaml_file in yaml_files:
  print(f"processing {yaml_file}")
  with open(yaml_file, 'r') as fh:
    obj = yaml.safe_load(fh)
  # fix up some fields:
  for production_line in obj["components"].get("ProducerComponent",{}).get("productionlines",{}).values():
    production_line["produces"] = {k: v for k, v in production_line.get("produces",[])} # convert to dict
    production_line["consumes"] = {k: v for k, v in production_line.get("consumes",[])}
  building_name = obj["id"].removeprefix("BUILDINGS.").removeprefix("UNITS.").lower()
  building_path = buildings_path / building_name
  building_path.mkdir(exist_ok=True)
  sprite_frames_animations = process_action_sets(obj["actionsets"], building_name, building_path)
  # display(obj)
  unique_source_atlas_paths = set(v["source_atlas_path"] for k, v in sprite_frames_animations.items())
  sprite_frames_tres_path = building_path / f"{building_name}.tres"
  obj["sprite_frames_tres_path"] = PurePosixPath(sprite_frames_tres_path.relative_to(project_path))
  obj["tscn_uid"] = generate_uid(obj["id"][0] + building_name, "tscn")
  obj["tres_uid"] = generate_uid(obj["id"][0] + building_name, "tres")
  if obj["tscn_uid"] in unique_uids or obj["tres_uid"] in unique_uids or obj["tscn_uid"] == obj["tres_uid"]:
    raise Exception(f"DUPLICATE UID: {obj['tscn_uid']}, {obj["tres_uid"]}")
  unique_uids.add(obj["tscn_uid"])
  unique_uids.add(obj["tres_uid"])
  obj["tscn_path"] = building_path / f"{building_name}.tscn"
  obj["tscn_res_path"] = PurePosixPath(obj["tscn_path"].relative_to(project_path))
  write_template(obj["tscn_path"], bakery_tscn_template, obj)
#   print(rt(bakery_tscn_template, obj))
  # print(rt(animation_frames_tscn_template, {"sprite_frames_animations": sprite_frames_animations, "uid": obj["tres_uid"]}))
  write_template(sprite_frames_tres_path, animation_frames_tscn_template, {"sprite_frames_animations": sprite_frames_animations, "uid": obj["tres_uid"]})
  buildings.append(obj)

processing S:\src\unknown-horizons\content\objects\units\animals\deer.yaml


idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\0.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\1.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\2.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\3.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\4.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\5.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\6.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\7.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\8.png
idle 000 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\0\9.png
idle 135 S:\src\unknown-horizons\content\gfx\units\animals\as_fallowdeer0\idle\135\0.png
idle 135 S:\src\unknown-horizons\content\

In [81]:
buildings_config_gd_template = """
extends Object
## The building config has all the information about the buildings required

class_name BuildingConfig

## All the buildings represented in enum state
enum Buildings {
  NONE                 =   0,
{% for building in buildings %}
  {{ building["str_id"].ljust(20, " ") }} = {{ building["enum_id"] }},
{% endfor %}
}

## Building enum value to the cost(resource to amount)
static var building_to_cost: Dictionary[Buildings, Dictionary] = {
  Buildings.NONE                : {},
{% for building in buildings %}
  Buildings.{{ building["str_id"].ljust(20, " ") }}: {{ building["buildingcosts"] | tojson | replace('"RES.', 'ResourceConfig.Resources.') | replace('":', ':') }},
{% endfor %}
}
"""
for i, building in enumerate(buildings):
  building["enum_id"] = i+100
  building["str_id"] = building["id"].removeprefix("BUILDINGS.").upper()

built_tileset_tres_template = """
{% for building in buildings %}
[ext_resource type="PackedScene" uid="uid://{{ building["tscn_uid"] }}" path="res://{{ building["tscn_res_path"] }}" id="2_{{ building["str_id"] }}"]
{% endfor %}

[sub_resource type="TileSetScenesCollectionSource" id="TileSetScenesCollectionSource_xv0cf"]
resource_name = "Buildings"
{% for building in buildings %}
scenes/{{ building["enum_id"] }}/scene = ExtResource("2_{{ building["str_id"] }}")
{% endfor %}
"""

# print(rt(buildings_config_gd_template, {"buildings": buildings}))
print(rt(built_tileset_tres_template, {"buildings": buildings}))


[ext_resource type="PackedScene" uid="uid://cambienttscn0" path="res://Assets/World/Buildings2/ambient/ambient.tscn" id="2_AMBIENT"]
[ext_resource type="PackedScene" uid="uid://cbakerytscn00" path="res://Assets/World/Buildings2/bakery/bakery.tscn" id="2_BAKERY"]
[ext_resource type="PackedScene" uid="uid://cbarrackstscn" path="res://Assets/World/Buildings2/barracks/barracks.tscn" id="2_BARRACKS"]
[ext_resource type="PackedScene" uid="uid://cbarriertscn0" path="res://Assets/World/Buildings2/barrier/barrier.tscn" id="2_BARRIER"]
[ext_resource type="PackedScene" uid="uid://cblendertscn0" path="res://Assets/World/Buildings2/blender/blender.tscn" id="2_BLENDER"]
[ext_resource type="PackedScene" uid="uid://cboatbuildets" path="res://Assets/World/Buildings2/boat_builder/boat_builder.tscn" id="2_BOAT_BUILDER"]
[ext_resource type="PackedScene" uid="uid://cbrewerytscn0" path="res://Assets/World/Buildings2/brewery/brewery.tscn" id="2_BREWERY"]
[ext_resource type="PackedScene" uid="uid://cbrickyard